In [ ]:
!git clone https://github.com/MIT-SPARK/Hydra-ROS.git
%cd Hydra-ROS

In [ ]:
# !git clone https://github.com/mit-han-lab/efficientvit.git
! git clone https://github.com/CVHub520/efficientvit
%cd efficientvit

In [ ]:

!pip install timm einops
!pip install onnx
!pip install onnxsim
!pip install triton
!pip install segment_anything
!pip install torchpack
!pip install pyngrok

In [ ]:
import torch
torch.backends.cudnn.benchmark = True
print(torch.cuda.get_device_name())

In [ ]:
mkdir -p assets/checkpoints/seg/ade20k


In [ ]:
!wget https://huggingface.co/han-cai/efficientvit-seg/resolve/main/efficientvit_seg_l2_ade20k.pt \
-O assets/checkpoints/seg/ade20k/l2.pt

In [ ]:
ls -lh assets/checkpoints/seg/ade20k/

In [ ]:
import random

import sys

sys.path.append("/content/efficientvit")

from efficientvit.seg_model_zoo import create_seg_model

torch.backends.cudnn.benchmark = True

# Load model
model = create_seg_model(
    # "efficientvit-seg-l2",
    "l2",
    dataset="ade20k",
    pretrained=True
)

model = model.cuda()
model.eval()


In [ ]:
def img_preprocessing(img):

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]

    target_w = 768
    target_h = 512

    scale = min(target_w / w, target_h / h)

    # new_w = int(w * scale)
    # new_h = int(h * scale)
    new_w = 512
    new_h = 320

    resized = cv2.resize(img, (new_w, new_h))

    padded = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    padded[:new_h, :new_w] = resized

    img = padded.astype(np.float32) / 255.0

    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    img = (img - mean) / std

    img_tensor = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).float().cuda()

    return img_tensor, (new_h, new_w)

import numpy as np
np.random.seed(42)
PALETTE = np.random.randint(0, 255, (256, 3), dtype=np.uint8)

def colorize_mask(mask):
  mask = cv2.resize(mask, (768, 512), interpolation=cv2.INTER_NEAREST)
  return PALETTE[mask]
  '''
  mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
  colors = np.random.randint(0,255,(256,3),dtype=np.uint8)
  colored = colors[mask]
  return colored
  '''

In [ ]:
from flask import Flask, request
import numpy as np
import cv2
import torch
import torch.nn.functional as F
# import sys
from pyngrok import ngrok
from google.colab.patches import cv2_imshow

ngrok.set_auth_token("3BpJprWOP9SUgUOd1HG7t9yJwZz_5kUK4cUC43Zs67ft77RRF")
public_url = ngrok.connect(5000)
print(public_url)

# sys.path.append("/content/efficientvit")

# from efficientvit.seg_model_zoo import create_seg_model

app = Flask(__name__)
'''
torch.backends.cudnn.benchmark = True

# Load model
model = create_seg_model(
    # "efficientvit-seg-l2",
    "l2",
    dataset="ade20k",
    pretrained=True
)

model = model.cuda()
model.eval()
'''

@app.route("/segment", methods=["POST"])
def segment():
    camera_source = request.form.get("source", "unknown")
    file = request.files["image"]
    img = np.frombuffer(file.read(), np.uint8)

    img = cv2.imdecode(img, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # print input image
    in_img = img
    print("input: ", in_img.shape)
    cv2_imshow(in_img)
    img_tensor, (new_h, new_w) = img_preprocessing(in_img)
    # left_cam | right_camera | seg_cam
    # resize image to multiple of 32
    '''
    h, w = img.shape[:2]
    new_h = (h // 32) * 32
    new_w = (w // 32) * 32
    img = cv2.resize(img, (new_w, new_h))

    img_tensor = torch.from_numpy(img).float()
    img_tensor = img_tensor.permute(2,0,1).unsqueeze(0)

    img_tensor = img_tensor.cuda()
    '''

    with torch.no_grad():
        output = model(img_tensor)
        output = F.interpolate(
        output,
        size=(512, 768),
        mode="bilinear",
        align_corners=False
    )
    # mask = output.argmax(1).cpu().numpy()[0].astype(np.uint8)
    mask = torch.argmax(output, dim=1)[0].cpu().numpy()

    mask = mask[:new_h, :new_w]

    mask = cv2.resize(
        mask,
        (in_img.shape[1], in_img.shape[0]),
        interpolation=cv2.INTER_NEAREST
    )
    print("Print unique labels:", np.unique(mask))
    # visualize coloured output image
    colored_mask = colorize_mask(mask)
    # vis = np.hstack([in_img, colored_mask])
    print("output: ", colored_mask.shape)
    cv2_imshow(colored_mask)

    _, encoded = cv2.imencode(".png", mask)

    return encoded.tobytes()


app.run(host="0.0.0.0", port=5000)